In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
# Base dataset
df = pd.read_csv("../Data/Processed/renttherunway_clean.csv")

# GMM clusters
gmm = pd.read_csv("../Data/Processed/gmm_df.csv")[["user_id", "item_id", "body_cluster", "cluster_confidence"]]

# DBSCAN clusters
dbscan = pd.read_csv("../Data/Processed/dbscan_df.csv")[["user_id", "dbscan_cluster"]]

# K-Means clusters
kmeans = pd.read_csv("../Data/Processed/kmeans_clusters.csv").rename(columns={"cluster": "kmeans_cluster"})

# LDA topic features
lda = pd.read_csv("../Data/Processed/renttherunway_lda_topics.csv")

In [ ]:
df = df.merge(
    dbscan,
    on="user_id",
    how="left"
)

In [ ]:
df = df.merge(
    gmm,
    on=["user_id", "item_id"],
    how="left"
)

In [ ]:
df = df.merge(
    kmeans,
    on=["user_id", "item_id"],
    how="left"
)

In [ ]:
df = df.merge(
    lda,
    on=["user_id", "item_id", "fit", "fit_label"],
    how="left"
)

In [ ]:
df["is_outlier"] = (df["dbscan_cluster"] == -1).astype(int)

In [ ]:
NUMERIC_FEATURES = [
    "height_inches",
    "weight_lbs",
    "bmi",
    "age",
    "bust_band",
    "cup_size_num"
]

CLUSTER_FEATURES = [
    "dbscan_cluster",
    "body_cluster",
    "kmeans_cluster"
]

BINARY_FEATURES = ["is_outlier"]

TOPIC_FEATURES = [c for c in df.columns if c.startswith("topic_")]

In [ ]:
X = df[
    NUMERIC_FEATURES
    + CLUSTER_FEATURES
    + BINARY_FEATURES
    + TOPIC_FEATURES
]

y = df["fit_label"].astype(int)

In [ ]:
assert y.notna().all()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        # Numeric + topic features
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            NUMERIC_FEATURES + TOPIC_FEATURES
        ),

        # Cluster features (categorical)
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            CLUSTER_FEATURES
        ),

        # Binary features
        (
            "bin",
            SimpleImputer(strategy="constant", fill_value=0),
            BINARY_FEATURES
        )
    ],
    remainder="drop"
)

In [ ]:
svm = SVC(
    kernel="rbf",
    probability=True,      # REQUIRED for predict_proba
    class_weight="balanced",
    random_state=42
)

In [ ]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", svm)
])

In [ ]:
param_grid = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.1, 0.01]
}

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

In [ ]:
best_svm = grid.best_estimator_

y_test_pred = best_svm.predict(X_test)

print("Best Kernel SVM parameters:")
print(grid.best_params_)

print("\nKernel SVM — Test Set Performance")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Small", "Fit", "Large"]
))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["Small", "Fit", "Large"],
    ax=ax,
    cmap="Blues",
    colorbar=False
)

plt.title("Kernel SVM — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("../figures/kernel_svm_confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
svm_proba = best_svm.predict_proba(X_test)

svm_proba_df = pd.DataFrame({
    "user_id": df.loc[X_test.index, "user_id"],
    "item_id": df.loc[X_test.index, "item_id"],
    "true_fit": y_test.values,
    "prob_small": svm_proba[:, 0],
    "prob_fit": svm_proba[:, 1],
    "prob_large": svm_proba[:, 2]
})

svm_proba_df.to_csv(
    "../Data/Processed/kernel_svm_fit_probabilities.csv",
    index=False
)

In [ ]:
prep = best_svm.named_steps["prep"]

feature_names = []
feature_names.extend(NUMERIC_FEATURES + TOPIC_FEATURES)

cat_encoder = prep.named_transformers_["cat"].named_steps["onehot"]
feature_names.extend(
    cat_encoder.get_feature_names_out(CLUSTER_FEATURES)
)

feature_names.extend(BINARY_FEATURES)

pd.Series(feature_names).to_csv(
    "../Data/Processed/kernel_svm_feature_names.csv",
    index=False
)

In [ ]:
import joblib

joblib.dump(
    best_svm,
    "../Models/kernel_svm_fit_classifier.joblib"
)